# BUILDING A SIMPLE TRANSFORMER MODEL FROM SCRATCH

In [1]:
import torch.nn as nn


class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        assert (self.head_dim * heads == embed_size), "Embedding size needs to be divisible by heads"
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]
        value_len = values.shape[1]
        key_len = keys.shape[1]
        query_len = query.shape[1]
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = query.reshape(N, query_len, self.heads, self.head_dim)
        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.head_dim ** 0.5), dim=3)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(N, query_len, self.embed_size)
        out = self.fc_out(out)
        return out

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(MultiHeadAttention, self).__init__()
        self.attention = SelfAttention(embed_size, heads)

    def forward(self, values, keys, query, mask):
        return self.attention(values, keys, query, mask)

In [3]:
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len, device):
        super().__init__()
        self.encoding = torch.zeros(max_len, embed_size, device=device)
        self.encoding.requires_grad = False
        pos = torch.arange(0, max_len, device=device).unsqueeze(1).float()
        two_i = torch.arange(0, embed_size, 2, device=device).float() / embed_size
        self.encoding[:, 0::2] = torch.sin(pos / (10000 ** two_i))
        self.encoding[:, 1::2] = torch.cos(pos / (10000 ** two_i))

    def forward(self, x):
        return x + self.encoding[:x.shape[1], :]

In [4]:
import torch.nn as nn


class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, forward_expansion):
        super().__init__()
        self.attention = MultiHeadAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.feed_forward = nn.Sequential(
                nn.Linear(embed_size, forward_expansion * embed_size), nn.ReLU(),
                nn.Linear(forward_expansion * embed_size, embed_size)
        )

    def forward(self, value, key, query, mask):
        attention = self.attention(value, key, query, mask)
        x = self.norm1(attention + query)
        forward = self.feed_forward(x)
        out = self.norm2(forward + x)
        return out

In [5]:
import torch.nn as nn


class Transformer(nn.Module):
    def __init__(self, embed_size, num_layers, heads, device, forward_expansion, max_length):
        super().__init__()
        self.embed_size = embed_size
        self.device = device
        self.positional_encoding = PositionalEncoding(embed_size, max_length, device)
        self.layers = nn.ModuleList([TransformerBlock(embed_size, heads, forward_expansion) for _ in range(num_layers)])

    def forward(self, x, mask):
        out = self.positional_encoding(x)

        for layer in self.layers:
            out = layer(out, out, out, mask)

        return out

# HANDS-ON: FINE-TUNING A PRETRAINED TRANSFORMER FOR TEXT CLASSIFICATION

In [6]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
train_data = dataset['train']
test_data = dataset['test']

In [8]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=256)


train_data = train_data.map(tokenize_function, batched=True)
test_data = test_data.map(tokenize_function, batched=True)

In [9]:
from transformers import TrainingArguments
from transformers import Trainer

training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        warmup_steps=500,
        weight_decay=0.01,
        eval_strategy="epoch",
        logging_dir="./logs"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=test_data
)

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.334699,0.391678
2,0.207986,0.368943
3,0.087427,0.403251


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9375, training_loss=0.22046904744466145, metrics={'train_runtime': 3903.4997, 'train_samples_per_second': 19.214, 'train_steps_per_second': 2.402, 'total_flos': 9866664576000000.0, 'train_loss': 0.22046904744466145, 'epoch': 3.0})

In [12]:
evaluation_results = trainer.evaluate()
print(evaluation_results)

Training Loss,Validation Loss,Epoch
0.087427,0.403251,3


{'eval_loss': 0.4032514989376068}


In [14]:
import torch

device = next(model.parameters()).device
sample_review = "This movie was absolutely fantastic, I loved every minute of it!"
inputs = tokenizer(sample_review, return_tensors="pt", padding=True, truncation=True, max_length=256)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predicted_class = torch.argmax(outputs.logits, dim=1).item()

if predicted_class == 1:
    print("Positive review")
else:
    print("Negative review")

Positive review


In [15]:
model.save_pretrained("./fine-tuned-bert")
tokenizer.save_pretrained("./fine-tuned-bert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine-tuned-bert/tokenizer_config.json', './fine-tuned-bert/tokenizer.json')

In [17]:
model = BertForSequenceClassification.from_pretrained("./fine-tuned-bert")
tokenizer = BertTokenizer.from_pretrained("./fine-tuned-bert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# STEP-BY-STEP GUIDE TO FINE-TUNING

In [18]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
train_data = dataset['train']
test_data = dataset['test']

In [20]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=256)


train_data = train_data.map(tokenize_function, batched=True)
test_data = test_data.map(tokenize_function, batched=True)

In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        warmup_steps=500,
        weight_decay=0.01,
        eval_strategy="epoch",
        logging_dir="./logs"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [22]:
from transformers import Trainer

trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=test_data
)

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.329283,0.255597
2,0.214856,0.332577
3,0.085737,0.412810


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9375, training_loss=0.22740168253580728, metrics={'train_runtime': 3655.7727, 'train_samples_per_second': 20.515, 'train_steps_per_second': 2.564, 'total_flos': 9866664576000000.0, 'train_loss': 0.22740168253580728, 'epoch': 3.0})

In [24]:
evaluation_results = trainer.evaluate()
print(evaluation_results)

Training Loss,Validation Loss,Epoch
0.085737,0.412810,3


{'eval_loss': 0.41281023621559143}


In [26]:
import torch

device = next(model.parameters()).device
sample_review = "This movie was absolutely fantastic, I loved every minute of it!"
inputs = tokenizer(sample_review, return_tensors="pt", padding=True, truncation=True, max_length=256)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predicted_class = torch.argmax(outputs.logits, dim=1).item()

if predicted_class == 1:
    print("Positive review")
else:
    print("Negative review")

Positive review


In [27]:
model.save_pretrained("./fine-tuned-bert")
tokenizer.save_pretrained("./fine-tuned-bert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine-tuned-bert/tokenizer_config.json', './fine-tuned-bert/tokenizer.json')

In [28]:
model = BertForSequenceClassification.from_pretrained("./fine-tuned-bert")
tokenizer = BertTokenizer.from_pretrained("./fine-tuned-bert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]